# DocStruct — 14-flag ablation sweep (Colab T4)

Upload this notebook, set **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

Runs the full config-flag ablation sweep from `scripts/_sweep.sh` — locally this is ~16h
(CPU-bound, and this environment kills long unattended jobs hourly), on a T4 it should be
well under an hour.

## It is meant to be run more than once

Free Colab reclaims sessions without warning. Everything expensive lives on Drive: the
corpus (PDFs + gold QA + YOLO weights), the detector-proposal cache, and every finished
ablation result. **If the session dies, reopen and Run all again** — nothing already on
Drive recomputes.

**Before running:** upload `zip/docstruct_corpus.zip` (983 MB — 102 PDFs, the merged
gold `benchmark_qa_v7.json`, and the YOLO weights) to your Google Drive. The path in
section 2 assumes Drive root; edit it if you put it elsewhere.

Output: `reports/ablations/ab_*.json` (one per flag) plus a comparison table you can paste
straight into `notes.md` and cite in the paper.

## 1. GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "NO GPU - Runtime > Change runtime type > T4 GPU, then Run all again."
print('GPU ok:', torch.cuda.get_device_name(0))

# Record the Pillow this kernel has ALREADY loaded, before anything pip-installs over
# it. Colab imports PIL at startup, and PIL._imaging is a compiled C extension: once it
# is in the process it cannot be unloaded, not by deleting sys.modules entries and not
# by reimporting. If pip later upgrades the Python files under it, this kernel keeps the
# old PIL._typing in memory against the new modules, and the mismatch surfaces as
# ImportError: cannot import name '_Ink' from 'PIL._typing' the moment ultralytics loads
# the model. Pinning pillow back to this version after install (section 3) means the
# on-disk files never actually change, so there is nothing to mismatch.
import PIL
PRELOADED_PILLOW = PIL.__version__
print('kernel loaded pillow', PRELOADED_PILLOW, '- will pin back to this after install')

## 2. Drive — every expensive artefact lives here

`BENCH` is the single directory this notebook owns. Deleting it resets everything;
leaving it alone is what makes a re-run resume.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS if you uploaded the zip somewhere other than Drive root
CORPUS_ZIP = '/content/drive/MyDrive/docstruct_corpus.zip'

# Gold ships in the repo, not the zip: it is small, and it is the one artefact
# that must match the code rather than the upload.
QA = 'data/qa/benchmark_qa_v9.json'

BENCH    = '/content/drive/MyDrive/docstruct_bench'
CORPUS   = BENCH + '/corpus'         # unzipped once, reused every session
CACHE    = BENCH + '/.bench_cache'   # detector proposals, shared across all 14 runs
REPORTS  = BENCH + '/reports/ablations'
for d in (BENCH, CORPUS, CACHE, REPORTS):
    os.makedirs(d, exist_ok=True)

assert os.path.exists(CORPUS_ZIP), f"not found: {CORPUS_ZIP} - upload it to Drive first and fix the path above"
print('corpus zip ->', CORPUS_ZIP, round(os.path.getsize(CORPUS_ZIP) / 1e6), 'MB')
print('bench dir  ->', BENCH)

## 3. Repo and dependencies

**Ignore pip's dependency-conflict warnings** about `jedi`, `requests`/`google-colab` and
`opentelemetry`. Pip reports on the whole environment, including Colab preinstalls this
run never imports. Only a traceback matters.

The one that *is* fatal is Pillow — section 1 recorded the version this kernel loaded at
startup, and the cell below pins it straight back after install (see section 1 for why a
restart, which `Run all` cannot do, would otherwise be required).

In [ ]:
%cd /content
if not os.path.exists('/content/DocStruct/.git'):
    !git clone -q -b feat/paper-draft https://github.com/CandyButcher27/DocStruct
%cd /content/DocStruct
!git pull -q --ff-only 2>/dev/null || echo '(could not fast-forward; keeping what is here)'
!git log --oneline -1

## 4. Install

`retrieval` (embeddings), `benchmark` (comparison adapters — ablate.py only scores
`docstruct`, but importing the eval package touches all of them), `model` (YOLO
detector, GPU-accelerated here).

In [ ]:
!pip install -q -e ".[retrieval,benchmark,model]"

# Put Pillow back to the version this kernel loaded at startup (see section 1). The
# alternative -- upgrading and restarting the runtime -- cannot be done inside Run all.
!pip install -q "pillow=={PRELOADED_PILLOW}"

import PIL
from PIL import Image
Image.new('RGB', (4, 4))
assert PIL.__version__ == PRELOADED_PILLOW, (
    f'pillow is {PIL.__version__}, kernel loaded {PRELOADED_PILLOW} -- the C extension and the '
    'Python files disagree. Restart the runtime and Run all again.'
)
print('pillow', PIL.__version__, 'ok (core and files agree)')

In [ ]:
# Ultralytics ships a top-level `tests` package that shadows ours, and importing YOLO is
# what puts it on the path. Confirm the real PIL survives an ultralytics import before the
# long run rather than partway through the first ablation.
from ultralytics import YOLO
from PIL import Image
Image.new('RGB', (4, 4))
print('ultralytics imported, PIL still alive')

# Which adapters actually imported. get_adapters() swallows import errors and drops
# anything whose available() is False, so a missing dependency would silently shrink
# things instead of failing loudly. ablate.py only needs 'docstruct'.
from docstruct.eval.adapters import get_adapters
got = get_adapters(names=['docstruct'], weights=None)
assert 'docstruct' in got, "docstruct adapter did not import - check the traceback above"
print('docstruct adapter: ok')

## 5. Corpus — unpacked once, cached on Drive

Unzips `docstruct_corpus.zip` into `CORPUS` on Drive the first time only, then copies
from there into the local repo every session (local disk is what the runs read — the
benchmark opens every PDF repeatedly, and Drive's FUSE mount is slow for that). This
overwrites `data/qa/` with the merged gold (`benchmark_qa_v7.json`, includes the
recovered docs) and fills in `data/raw-pdfs/` and `weights/`, both gitignored so they
don't come from the clone.

In [ ]:
if not os.path.exists(CORPUS + '/data/raw-pdfs'):
    print('first run: unzipping corpus to Drive (~1GB, one-time)')
    !unzip -q -o "$CORPUS_ZIP" -d "$CORPUS"
else:
    print('corpus already unzipped on Drive, skipping')

# PDFs and weights come from the zip; gold does NOT. The zip predates the corpus
# repair, so its data/qa/ still holds gold written against papers that were later
# replaced -- copying it over would silently restore the broken pairing.
!cp -rn "{CORPUS}/data/raw-pdfs" data/
!mkdir -p weights && cp -n "{CORPUS}/weights/"*.pt weights/

import glob
n_pdfs = len(glob.glob('data/raw-pdfs/*.pdf'))
print('raw-pdfs:', n_pdfs, '(expect 102)')
print('gold (from the git clone, not the zip):', QA)
assert os.path.exists(QA), f'{QA} missing -- git pull, it is committed to the repo'
assert n_pdfs >= 100, f'only {n_pdfs} PDFs unpacked - corpus zip may be incomplete'
!ls weights


## 6. Smoke — before scaling

Per project convention: never run the full sweep before confirming one doc works end to
end, on the exact path the sweep uses. Read the per-doc output, not just the exit code —
`benchmark_tool` catches per-document exceptions internally, so a config that breaks
chunking on every doc can still exit 0 with an all-errors row.

In [ ]:
import glob
one = sorted(glob.glob('data/raw-pdfs/*.pdf'))[0]
print('smoke-testing', one)
!python -m docstruct.cli run "$one"

In [ ]:
# One baseline ablation run, same params + cache dir the sweep will use, on 3 docs only.
!python scripts/ablate.py --name ab_smoketest --qa "{QA}" --cache-dir "{CACHE}" --limit-docs 3

import json
# ablate.py nests every metric under "metrics"; the top level holds name/config/per_doc.
m = json.load(open('reports/ablations/ab_smoketest.json'))['metrics']
print('MRR:', m['mrr'], '| errors:', m['errors'], '| questions:', m['n_questions'])
assert m['errors'] == 0, f"{m['errors']} errors on the smoke docs - fix before the full sweep"
assert m['n_questions'] > 0, '0 questions scored - the gold does not match these PDFs; the sweep would measure nothing'
assert m['mrr'] > 0, 'MRR is 0 on the smoke docs - something is broken, not just unlucky'
print('smoke test ok')


## 7. The sweep — resumable, 14 flags

Same flags as `scripts/_sweep.sh`. Each run's result is copied to `REPORTS` on Drive the
moment it finishes, not at the end — so a reclaimed session loses at most the flag in
flight. Re-running this cell (or the whole notebook) skips any flag whose result is
already on Drive and restores it instead. The detector-proposal cache (`CACHE`, also on
Drive) is shared and warm across all 14 runs — only the very first run anywhere pays the
full YOLO detection cost.

In [ ]:
import os, shutil, subprocess, sys, json

os.makedirs('reports/ablations', exist_ok=True)

RUNS = [
    ("baseline", None),
    ("dedupe", "DEDUPE_CHARS=True"),
    ("dehyphen", "DEHYPHENATE=True"),
    ("normalize", "NORMALIZE_TEXT=True"),
    ("fig_area", "FIGURE_OVERLAP_BY_AREA=True"),
    ("multicol", "MULTI_COLUMN=True"),
    ("bandsplit", "BAND_SPLIT=True"),
    ("furniture", "STRIP_PAGE_FURNITURE=True"),
    ("tbl_borderless", "TABLE_TEXT_STRATEGY_FALLBACK=True"),
    ("tbl_keyvalue", "TABLE_SERIALIZATION=keyvalue"),
    ("tbl_split", "TABLE_SPLIT_ROWS=True"),
    ("hdr_bold", "HEADER_RANK_BY_WEIGHT=True"),
    ("keep_refs", "KEEP_REFERENCES=True"),
    ("label_contain", "LABEL_AWARE_CONTAINMENT=True"),
]

for flag, override in RUNS:
    name = f"ab_{flag}"
    drive_json = f"{REPORTS}/{name}.json"
    local_json = f"reports/ablations/{name}.json"

    if os.path.exists(drive_json):
        print(f"== {name}: already on Drive, restoring and skipping")
        shutil.copy(drive_json, local_json)
        continue

    print(f"== {name}: running ({override or 'no override'})", flush=True)
    args = [sys.executable, "scripts/ablate.py", "--name", name, "--qa", QA,
            "--cache-dir", CACHE]
    if override:
        args += ["--set", override]
    proc = subprocess.run(args, capture_output=True, text=True)
    print("\n".join(proc.stdout.splitlines()[-12:]))
    if proc.returncode != 0:
        print(proc.stderr[-2000:])
        raise SystemExit(f"{name} failed - fix before continuing the sweep")

    shutil.copy(local_json, drive_json)
    print(f"   -> saved to Drive: {drive_json}")

print("\n=== SWEEP DONE ===")

## 8. Compare every flag against baseline

In [ ]:
import json, glob

def metrics(path):
    d = json.load(open(path))
    return d['name'], d['metrics']          # metrics are nested, not top-level

_, base = metrics('reports/ablations/ab_baseline.json')
base_mrr = base['mrr']

rows = []
for f in sorted(glob.glob('reports/ablations/ab_*.json')):
    name, m = metrics(f)
    if name in ('ab_baseline', 'ab_smoketest'):
        continue
    rows.append((name, m['mrr'], m['mrr'] - base_mrr, m['ndcg'], m['recall'], m['errors']))

rows.sort(key=lambda r: -r[2])

print(f"{'flag':22} {'MRR':>8} {'delta':>8} {'NDCG':>8} {'Recall':>8} {'errors':>7}")
print(f"{'baseline':22} {base_mrr:8.4f} {'--':>8} {base['ndcg']:8.4f} {base['recall']:8.4f} {base['errors']:7}")
for name, mrr, delta, ndcg, recall, err in rows:
    flag = 'WIN' if delta > 0.001 else ('LOSS' if delta < -0.001 else 'flat')
    print(f"{name:22} {mrr:8.4f} {delta:+8.4f} {ndcg:8.4f} {recall:8.4f} {err:7}  {flag}")

print()
print('baseline questions scored:', base['n_questions'], '| chunks:', base['n_chunks'])


Copy the table above into `notes.md`, and for any flag marked `WIN`, flip its default in `config.py` — with a comment naming this run as the justification (project rule 5).

## 9. Collect for download

Everything is already on Drive (`REPORTS`, saved incrementally in section 7). This is
just the convenience copy for a local zip you can download from the browser.

In [ ]:
!cd reports && zip -q -r /content/docstruct_ablations.zip ablations/*.json
try:
    from google.colab import files
    files.download('/content/docstruct_ablations.zip')
except Exception as e:
    print('browser download skipped:', e)
    print(f'the results are on Drive at {REPORTS}')